In [17]:

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
import faiss

import warnings
warnings.filterwarnings("ignore")

In [18]:
FILE_PATH = "usa.txt"   

loader = TextLoader(FILE_PATH, encoding="utf-8")
documents = loader.load()

print(f"Loaded {len(documents)} document(s)")


Loaded 1 document(s)


In [19]:
#  Split into Chunks

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

split_docs = splitter.split_documents(documents)


In [20]:

# Add chunk_id metadata(since no page numbers in TXT)

for i, doc in enumerate(split_docs):
    doc.metadata["chunk_id"] = i

print(f"Split into {len(split_docs)} chunks")


Split into 13 chunks


In [21]:

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1922.91it/s]


In [22]:
#  Create FAISS Index

embedding_dim = 384

index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

vector_store.add_documents(split_docs)

print("Documents embedded and indexed")


Documents embedded and indexed


In [23]:
#  Retriever

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

In [24]:
#  LLM (Ollama)

llm = ChatOllama(model="llama3.2")

In [25]:
#  Query

query = "USA GDP in 2024?"

In [26]:
# Retrieve Documents

retrieved_docs = retriever.invoke(query)

In [27]:
#  Build Context

context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

In [28]:
#  Prompt Template

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Use ONLY the provided context to answer.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{question}

Answer:
""")

formatted_prompt = prompt.format(
    context=context_text,
    question=query
)

In [29]:
#  LLM Response

response = llm.invoke(formatted_prompt)

print("\n Answer:\n", response.content)


 Answer:
 The United States' nominal GDP in 2024 is estimated to be around $28 trillion USD.


In [30]:
# Show Retrieved Chunks

print("\n Retrieved Chunks:\n")

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"--- Chunk {i} ---")
    print(f"Chunk ID: {doc.metadata.get('chunk_id', 'N/A')}")
    print(doc.page_content[:400], "...\n")


 Retrieved Chunks:

--- Chunk 1 ---
Chunk ID: 1
U.S. GDP – Size, Composition, and Global Share
As of 2024, the United States’ nominal GDP is estimated to be around $28 trillion USD, accounting for approximately 25% of the global economy. It ranks #1 in the world by nominal GDP, far ahead of China (which ranks 2nd). The U.S. GDP per capita is also among the highest, hovering around $83,000, which indicates a high standard of living and productiv ...

--- Chunk 2 ---
Chunk ID: 3
GDP Growth Trends and Dynamics
Historically, the U.S. economy has enjoyed consistent long-term growth, averaging around 2-3% annually. Post-pandemic, the economy bounced back strongly, but 2022 and 2023 saw rising inflation due to supply chain issues and stimulus-driven demand. In 2024, the U.S. GDP grew at a modest pace of around 2.1%, as the Federal Reserve’s interest rate hikes aimed at control ...

--- Chunk 3 ---
Chunk ID: 12
Final Summary
The U.S. economy remains the engine of global growth, backed by unma